# Feature Engineering

Derive modeling-ready features from the integrated flights + weather table.

## Input
- `data/processed/integrated/flights_2024_weather.parquet`

## Output
- `data/processed/integrated/features_2024.parquet`

## Feature categories
1. **Time features** — `dep_hour`, `month`, `day_of_week`, `is_weekend`, `time_block`
2. **Holiday features** — `is_holiday`, `holiday_proximity`
3. **Hub / distance features** — `is_origin_hub`, `is_dest_hub`, `distance_bin`
4. **Rolling historical features** — `airline_delay_rate_7d`, `route_delay_rate_7d`, `origin_delay_rate_7d`, `origin_daily_flights` (all shifted to prevent leakage)
5. **Cascading delay** — `prev_flight_arr_delay` (same aircraft's prior flight)
6. **Aircraft utilization** — `tail_leg_today` (which leg of the day for this aircraft)
7. **Hourly congestion** — `origin_hourly_flights` (departures from same airport in same hour)
8. **Weather interactions** — `origin/dest_freezing_rain`, `origin/dest_wind_rain`, `origin/dest_fog_risk`

## Step 0: Ensure project data (local or Hugging Face)

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
for _p in [_here] + list(_here.parents):
    if (_p / "notebooks" / "project_data.py").exists():
        sys.path.insert(0, str(_p / "notebooks"))
        break

from project_data import ensure_project_data

ensure_project_data()

## Step 1: Load integrated data

In [ ]:
import os, warnings
from pathlib import Path

import numpy as np
import pandas as pd

from project_data import resolve_project_root

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = resolve_project_root()
DATA_ROOT = Path(os.getenv("FLIGHT_DATA_DIR", PROJECT_ROOT / "data")).expanduser().resolve()
INTEGRATED_DIR = DATA_ROOT / "processed" / "integrated"

input_path = INTEGRATED_DIR / "flights_2024_weather.parquet"
if not input_path.exists():
    raise FileNotFoundError(f"Missing input: {input_path}")

df = pd.read_parquet(input_path)
df["FlightDate"] = pd.to_datetime(df["FlightDate"], errors="coerce")

initial_rows = len(df)
print(f"Loaded {initial_rows:,} rows, {len(df.columns)} columns")

## Step 2: Time features

Extract directly from `FlightDate` and `CRSDepTime`.

In [ ]:
dep_hhmm = pd.to_numeric(df["CRSDepTime"], errors="coerce")
df["dep_hour"] = (dep_hhmm // 100).clip(0, 23).astype("Int64")

df["month"] = df["FlightDate"].dt.month.astype("Int64")
df["day_of_week"] = df["FlightDate"].dt.dayofweek.astype("Int64")  # 0=Mon, 6=Sun
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

def hour_to_block(h):
    if 5 <= h <= 11:
        return "Morning"
    elif 12 <= h <= 16:
        return "Afternoon"
    elif 17 <= h <= 21:
        return "Evening"
    else:
        return "Night"

df["time_block"] = df["dep_hour"].map(hour_to_block).astype("category")

print("Time features created:")
print(f"  dep_hour range: {df['dep_hour'].min()} – {df['dep_hour'].max()}")
print(f"  month range:    {df['month'].min()} – {df['month'].max()}")
print(f"  is_weekend:     {df['is_weekend'].mean()*100:.1f}%")
print(f"  time_block dist:")
print(df["time_block"].value_counts().sort_index().to_string(header=False))

## Step 3: Holiday features

US federal holidays for 2024, with ±1 day buffer for travel surge.

In [ ]:
US_HOLIDAYS_2024 = pd.to_datetime([
    "2024-01-01",  # New Year's Day
    "2024-01-15",  # MLK Day
    "2024-02-19",  # Presidents' Day
    "2024-05-27",  # Memorial Day
    "2024-06-19",  # Juneteenth
    "2024-07-04",  # Independence Day
    "2024-09-02",  # Labor Day
    "2024-10-14",  # Columbus Day
    "2024-11-11",  # Veterans Day
    "2024-11-28",  # Thanksgiving
    "2024-12-25",  # Christmas
])

# Expand to ±1 day buffer
holiday_window = set()
for h in US_HOLIDAYS_2024:
    for delta in [-1, 0, 1]:
        holiday_window.add(h + pd.Timedelta(days=delta))

df["is_holiday"] = df["FlightDate"].isin(holiday_window).astype(int)

# Distance in days to the nearest holiday
holiday_arr = US_HOLIDAYS_2024.values
flight_dates = df["FlightDate"].values
diffs = np.abs(flight_dates[:, None] - holiday_arr[None, :]).astype("timedelta64[D]").astype(int)
df["holiday_proximity"] = diffs.min(axis=1)

print(f"is_holiday rate: {df['is_holiday'].mean()*100:.2f}%")
print(f"holiday_proximity: mean={df['holiday_proximity'].mean():.1f}, median={df['holiday_proximity'].median():.0f}")

## Step 4: Hub and distance features

In [ ]:
HUB_AIRPORTS = {
    "ATL", "ORD", "DFW", "DEN", "CLT",  # major connecting hubs
    "IAH", "EWR", "SFO", "LAX", "PHX",
    "MSP", "DTW", "JFK", "SEA", "BOS",
    "SLC", "IAD", "DCA", "LGA", "MIA",
}

df["is_origin_hub"] = df["Origin"].isin(HUB_AIRPORTS).astype(int)
df["is_dest_hub"] = df["Dest"].isin(HUB_AIRPORTS).astype(int)

dist_bins = [0, 500, 1000, 1500, 2000, np.inf]
dist_labels = ["<500mi", "500-1000mi", "1000-1500mi", "1500-2000mi", ">2000mi"]
df["distance_bin"] = pd.cut(
    pd.to_numeric(df["Distance"], errors="coerce"),
    bins=dist_bins,
    labels=dist_labels,
    right=True,
).astype("category")

print(f"is_origin_hub rate: {df['is_origin_hub'].mean()*100:.1f}%")
print(f"is_dest_hub rate:   {df['is_dest_hub'].mean()*100:.1f}%")
print(f"\ndistance_bin distribution:")
print(df["distance_bin"].value_counts().sort_index().to_string(header=False))

## Step 5: Rolling historical features (7-day delay rates)

Aggregate to daily level first, compute rolling means with `shift(1)` to prevent target leakage, then merge back to flight level.

- `airline_delay_rate_7d`: airline's delay rate over the past 7 days
- `origin_delay_rate_7d`: origin airport's delay rate over the past 7 days
- `route_delay_rate_7d`: route (Origin→Dest) delay rate over the past 7 days
- `origin_daily_flights`: number of flights departing from origin on that day

In [ ]:
df["Reporting_Airline"] = df["Reporting_Airline"].astype(str)
df["Origin"] = df["Origin"].astype(str)
df["Dest"] = df["Dest"].astype(str)

# --- Airline daily delay rate ---
airline_daily = (
    df.groupby(["Reporting_Airline", "FlightDate"])["DepDel15"]
    .mean()
    .rename("airline_delay_rate_daily")
    .reset_index()
    .sort_values("FlightDate")
)
airline_daily["airline_delay_rate_7d"] = (
    airline_daily.groupby("Reporting_Airline")["airline_delay_rate_daily"]
    .transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())
)
df = df.merge(
    airline_daily[["Reporting_Airline", "FlightDate", "airline_delay_rate_7d"]],
    on=["Reporting_Airline", "FlightDate"],
    how="left",
)

# --- Origin airport daily delay rate ---
origin_daily = (
    df.groupby(["Origin", "FlightDate"])
    .agg(origin_delay_rate_daily=("DepDel15", "mean"), origin_daily_flights=("DepDel15", "count"))
    .reset_index()
    .sort_values("FlightDate")
)
origin_daily["origin_delay_rate_7d"] = (
    origin_daily.groupby("Origin")["origin_delay_rate_daily"]
    .transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())
)
df = df.merge(
    origin_daily[["Origin", "FlightDate", "origin_delay_rate_7d", "origin_daily_flights"]],
    on=["Origin", "FlightDate"],
    how="left",
)

# --- Route daily delay rate ---
route_daily = (
    df.groupby(["Origin", "Dest", "FlightDate"])["DepDel15"]
    .mean()
    .rename("route_delay_rate_daily")
    .reset_index()
    .sort_values("FlightDate")
)
route_daily["route_delay_rate_7d"] = (
    route_daily.groupby(["Origin", "Dest"])["route_delay_rate_daily"]
    .transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())
)
df = df.merge(
    route_daily[["Origin", "Dest", "FlightDate", "route_delay_rate_7d"]],
    on=["Origin", "Dest", "FlightDate"],
    how="left",
)

print("Rolling features created:")
for col in ["airline_delay_rate_7d", "origin_delay_rate_7d", "route_delay_rate_7d", "origin_daily_flights"]:
    na_pct = df[col].isna().mean() * 100
    print(f"  {col:30s} mean={df[col].mean():.4f}  NaN={na_pct:.2f}%")

## Step 6: Cascading delay — previous flight arrival delay

For each aircraft (`Tail_Number`), sort by departure time and take `ArrDelay` from the immediately preceding flight via `shift(1)`. Flights with `UNKNOWN` tail number get NaN.

In [ ]:
# Build a sort key: FlightDate + dep_hour (approximate chronological order)
dep_hhmm = pd.to_numeric(df["CRSDepTime"], errors="coerce")
df["_sort_time"] = df["FlightDate"] + pd.to_timedelta((dep_hhmm // 100).clip(0, 23), unit="h")

df = df.sort_values(["Tail_Number", "_sort_time"]).reset_index(drop=True)

# shift(1) within each tail number
df["prev_flight_arr_delay"] = (
    df.groupby("Tail_Number")["ArrDelay"]
    .shift(1)
)

# Null out for UNKNOWN tail numbers (no reliable link)
df.loc[df["Tail_Number"] == "UNKNOWN", "prev_flight_arr_delay"] = np.nan

df = df.drop(columns=["_sort_time"])

valid_pct = df["prev_flight_arr_delay"].notna().mean() * 100
print(f"prev_flight_arr_delay coverage: {valid_pct:.1f}%")
print(f"  mean (when available): {df['prev_flight_arr_delay'].mean():.2f} min")

## Step 7: Aircraft daily utilization (`tail_leg_today`)

Which leg of the day is this flight for the same aircraft? Later legs accumulate more cascading delays.

In [ ]:
df = df.sort_values(["Tail_Number", "FlightDate", "CRSDepTime"]).reset_index(drop=True)

df["tail_leg_today"] = df.groupby(["Tail_Number", "FlightDate"]).cumcount()

# UNKNOWN tail numbers: set to NaN (no reliable utilization info)
df.loc[df["Tail_Number"] == "UNKNOWN", "tail_leg_today"] = np.nan

print("tail_leg_today distribution:")
print(df["tail_leg_today"].value_counts().sort_index().head(8).to_string(header=False))
print(f"\nDelay rate by leg:")
for leg in range(6):
    sub = df[df["tail_leg_today"] == leg]
    if len(sub) > 1000:
        print(f"  leg {leg}: {sub['DepDel15'].mean()*100:.1f}%  (n={len(sub):,})")

## Step 8: Hourly congestion (`origin_hourly_flights`)

Number of scheduled departures from the same origin airport in the same hour. Finer-grained than `origin_daily_flights`.

In [ ]:
hourly_counts = (
    df.groupby(["Origin", "FlightDate", "dep_hour"])
    .size()
    .rename("origin_hourly_flights")
    .reset_index()
)
df = df.merge(hourly_counts, on=["Origin", "FlightDate", "dep_hour"], how="left")

print("origin_hourly_flights:")
print(f"  mean={df['origin_hourly_flights'].mean():.1f}, median={df['origin_hourly_flights'].median():.0f}, "
      f"p95={df['origin_hourly_flights'].quantile(0.95):.0f}, max={df['origin_hourly_flights'].max():.0f}")
print(f"  NaN: {df['origin_hourly_flights'].isna().mean()*100:.2f}%")

## Step 9: Weather interaction features

Combine weather variables to capture compound effects that individual features miss.

- `origin_freezing_rain`: temperature ≤ 2°C AND precipitation > 0 → de-icing delays
- `origin_wind_rain`: wind speed > 8 m/s AND precipitation > 0 → visibility degradation
- `origin_fog_risk`: air temperature − dew point < 2°C → high fog probability
- Same three features for destination airport

In [ ]:
# --- Origin weather interactions ---
df["origin_freezing_rain"] = (
    (df["origin_air_temp"] <= 2) & (df["origin_precip_1h"] > 0)
).astype(int)

df["origin_wind_rain"] = (
    (df["origin_wind_speed"] > 8) & (df["origin_precip_1h"] > 0)
).astype(int)

df["origin_fog_risk"] = (
    (df["origin_air_temp"] - df["origin_dew_point"]) < 2
).astype(int)

# --- Destination weather interactions ---
df["dest_freezing_rain"] = (
    (df["dest_air_temp"] <= 2) & (df["dest_precip_1h"] > 0)
).astype(int)

df["dest_wind_rain"] = (
    (df["dest_wind_speed"] > 8) & (df["dest_precip_1h"] > 0)
).astype(int)

df["dest_fog_risk"] = (
    (df["dest_air_temp"] - df["dest_dew_point"]) < 2
).astype(int)

print("Weather interaction features:")
for col in ["origin_freezing_rain", "origin_wind_rain", "origin_fog_risk",
            "dest_freezing_rain", "dest_wind_rain", "dest_fog_risk"]:
    rate = df[col].mean() * 100
    # delay rate when flag=1
    flagged = df[df[col] == 1]
    d_rate = flagged["DepDel15"].mean() * 100 if len(flagged) > 0 else 0
    print(f"  {col:25s} prevalence={rate:5.2f}%  delay_rate_when_true={d_rate:.1f}%")

## Step 10: Verify and save

In [ ]:
NEW_FEATURES = [
    # time
    "dep_hour", "month", "day_of_week", "is_weekend", "time_block",
    # holiday
    "is_holiday", "holiday_proximity",
    # hub / distance
    "is_origin_hub", "is_dest_hub", "distance_bin",
    # rolling historical
    "airline_delay_rate_7d", "origin_delay_rate_7d", "route_delay_rate_7d",
    "origin_daily_flights",
    # cascading
    "prev_flight_arr_delay",
    # aircraft utilization
    "tail_leg_today",
    # hourly congestion
    "origin_hourly_flights",
    # weather interactions
    "origin_freezing_rain", "origin_wind_rain", "origin_fog_risk",
    "dest_freezing_rain", "dest_wind_rain", "dest_fog_risk",
]

# Row count should be unchanged
assert len(df) == initial_rows, f"Row count mismatch: {len(df)} vs {initial_rows}"

print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)
print(f"Rows:     {len(df):,} (unchanged)")
print(f"Columns:  {len(df.columns)} (was 58)")
print(f"New features: {len(NEW_FEATURES)}")
print()
for col in NEW_FEATURES:
    na_pct = df[col].isna().mean() * 100
    print(f"  {col:30s} dtype={str(df[col].dtype):12s} NaN={na_pct:.2f}%")
print("=" * 60)

out_path = INTEGRATED_DIR / "features_2024.parquet"
df.to_parquet(out_path, index=False)
print(f"\nSaved: {out_path}")
print(f"File size: {out_path.stat().st_size / 1024 / 1024:.1f} MB")